# Post-processing

In [7]:
import polars as pl
from pathlib import Path
from datetime import date

DATA_DIR = Path("../data/raw")

print("=== Расчет коэффициента сезонности на основе 2025 года ===")

data = pl.read_parquet(DATA_DIR / 'train.parquet', columns=['user_id', 'event_date', 'gmv'])

purchases = data.filter(pl.col("gmv") > 0)

base_period = purchases.filter(
    (pl.col("event_date") >= date(2025, 1, 14)) & 
    (pl.col("event_date") <= date(2025, 2, 13))
)

holiday_period = purchases.filter(
    (pl.col("event_date") >= date(2025, 2, 14)) & 
    (pl.col("event_date") <= date(2025, 3, 15))
)

base_mean_gmv = base_period["gmv"].mean()
holiday_mean_gmv = holiday_period["gmv"].mean()

total_users = data["user_id"].n_unique()

base_conversion = base_period["user_id"].n_unique() / total_users
holiday_conversion = holiday_period["user_id"].n_unique() / total_users

print(f"Средний чек (Базовый месяц): {base_mean_gmv:.2f}")
print(f"Средний чек (Праздничный месяц): {holiday_mean_gmv:.2f}")
print("-" * 40)
print(f"Конверсия (Базовый месяц): {base_conversion * 100:.2f}%")
print(f"Конверсия (Праздничный месяц): {holiday_conversion * 100:.2f}%")
print("-" * 40)

expected_gmv_base = base_conversion * base_mean_gmv
expected_gmv_holiday = holiday_conversion * holiday_mean_gmv

magic_multiplier = expected_gmv_holiday / expected_gmv_base

print(f"РАСЧЕТНЫЙ МАГИЧЕСКИЙ МНОЖИТЕЛЬ: {magic_multiplier:.4f}")

=== Расчет коэффициента сезонности на основе 2025 года ===


Средний чек (Базовый месяц): 59.53
Средний чек (Праздничный месяц): 59.97
----------------------------------------
Конверсия (Базовый месяц): 37.23%
Конверсия (Праздничный месяц): 40.29%
----------------------------------------
РАСЧЕТНЫЙ МАГИЧЕСКИЙ МНОЖИТЕЛЬ: 1.0901


In [7]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")
best_sub = pd.read_csv(PROCESSED / "v6_arithmetic_cb70_mlp20_xgb10.csv")

multipliers = [1.12375]

print("=== Генерация финальных сабмитов (Target: ~1.09) ===")
for m in multipliers:
    new_sub = best_sub.copy()
    new_sub["predict"] = new_sub["predict"] * m
    
    name = f"v6_champion_mult_{m:.5f}.csv"
    new_sub.to_csv(PROCESSED / name, index=False)
    print(f"Готов к отправке: {name} | Новый mean = {new_sub['predict'].mean():.2f}")

=== Генерация финальных сабмитов (Target: ~1.09) ===
Готов к отправке: v6_champion_mult_1.12375.csv | Новый mean = 39.83


In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")
champ_sub = pd.read_csv(PROCESSED / "v6_champion_mult_1.12375.csv")

gammas = [1.003, 1.007, 1.011, 1.015, 1.020]

print("=== Генерация 5 финальных сабмитов (Гамма) ===")
for g in gammas:
    sub = champ_sub.copy()
    sub["predict"] = sub["predict"] ** g
    
    name = f"v6_gamma_{g}.csv"
    sub.to_csv(PROCESSED / name, index=False)
    print(f"Готов: {name:<18} | Mean: {sub['predict'].mean():.2f} | Max: {sub['predict'].max():.2f}")

=== Генерация 5 финальных сабмитов (Гамма) ===
Готов: v6_gamma_1.003.csv | Mean: 40.43 | Max: 5440.27
Готов: v6_gamma_1.007.csv | Mean: 41.25 | Max: 5630.13
Готов: v6_gamma_1.011.csv | Mean: 42.09 | Max: 5826.61
Готов: v6_gamma_1.015.csv | Mean: 42.94 | Max: 6029.95
Готов: v6_gamma_1.02.csv  | Mean: 44.04 | Max: 6294.14
